In [1]:
import sys
sys.path.append("../")

##### import library

In [2]:
from typing import List, Optional
import requests
import pandas as pd
import mygene
import utility
import importlib
importlib.reload(utility)
from utility import resolve_ensembl_ids_with_fallback, get_chembl_ids_fast,  get_disease_ids_fast, validate_id_dataframe
import pickle

In [3]:
dct_result = dict()

##### Read pickle

In [4]:
with open("TTD_same_question_response_final.pkl", "rb") as f:
    TTD_same_question_response = pickle.load(f)


In [5]:
for model in ['llama-3.3-70b-versatile', 'gpt-5-nano', 'grok-4-1-fast-non-reasoning-latest', "gemini-2.5-flash-lite", "OpenTarget", "BioChatter", "ctd", "ttd", "hcdt"]:

    
    # if model=="llama-3.3-70b-versatile":
    #     dct_result[model] = Llama_same_question_response[model]

    # if model=="gpt-5-nano":
    #     dct_result[model] = OpenAI_same_question_response[model]

    # if model=='grok-4-1-fast-non-reasoning-latest':
    #     dct_result[model] = Grok_same_question_response[model]

    # if model=="gemini-2.5-flash-lite":
    #     dct_result[model] = Gemini_same_question_response[model]

    # if model=="OpenTarget":
    #     dct_result[model] = Opentarget_same_question_response[model]

    # if model=="BioChatter":
    #     dct_result[model] = Biochatter_same_question_response[model]

    # if model=="ctd":
    #     dct_result[model] = CTD_same_question_response[model]

    # if model=="hcdt":
    #     dct_result[model] = HCDT_same_question_response[model]

    if model=="ttd":
        dct_result[model] = TTD_same_question_response[model]


##### Verify if only valid columns are present

In [6]:
allowed = {"gene_name", "drug_name", "disease_name", "pathway_name", "target_name"}

missing_df = []
not_dataframe = []
empty_df = []
bad_columns = []

total_payloads = 0

for model, queries in dct_result.items():
    if not isinstance(queries, dict):
        continue

    for question, runs in queries.items():
        if not isinstance(runs, dict):
            continue

        for run_id, payload in runs.items():
            total_payloads += 1

            if not isinstance(payload, dict) or "dataframe" not in payload:
                missing_df.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "payload_type": type(payload).__name__
                })
                continue

            df = payload.get("dataframe")

            if not isinstance(df, pd.DataFrame):
                not_dataframe.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "df_type": type(df).__name__
                })
                continue

            if df.empty:
                empty_df.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "columns": list(df.columns)
                })

            cols = {str(c) for c in df.columns}
            extra = sorted(cols - allowed)

            if extra:
                bad_columns.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "columns": list(df.columns),
                    "extra_columns": extra
                })

print(f"Total payloads checked: {total_payloads}")
print(f"Missing dataframe key: {len(missing_df)}")
print(f"Dataframe is not pd.DataFrame: {len(not_dataframe)}")
print(f"Empty dataframes: {len(empty_df)}")
print(f"Dataframes with disallowed columns: {len(bad_columns)}")

bad_columns_df = pd.DataFrame(bad_columns)
missing_df_df = pd.DataFrame(missing_df)
not_dataframe_df = pd.DataFrame(not_dataframe)
empty_df_df = pd.DataFrame(empty_df)

display(bad_columns_df)
display(missing_df_df)
display(not_dataframe_df)
display(empty_df_df)


Total payloads checked: 324
Missing dataframe key: 0
Dataframe is not pd.DataFrame: 0
Empty dataframes: 0
Dataframes with disallowed columns: 0


""


""


""


""


In [7]:
# rows_missing_parentheses = []

# dct_jaccard = dict()

# for model, queries in dct_result.items():
#     dct_jaccard[model] = {}
#     print(f"\nWorking for model: {model}")

#     for question, runs in queries.items():

#         for run_id, payload in runs.items():
#             df = payload.get("dataframe")

#             # ---- validation ----
#             if not isinstance(df, pd.DataFrame) or df.empty:
#                 continue

#             # ---- skip pathway outputs ----
#             if "pathway_name" in df.columns:
#                 continue

#             # ---- column sanity ----
#             # if not set(df.columns).issubset(allowed_cols):
#             #     continue

#             # ======================================================
#             # CHECK ONLY target_name COLUMN
#             # ======================================================
#             if "target_name" in df.columns:

#                 mask = ~df["target_name"].astype(str).str.contains(
#                     r"\(.*\)", regex=True, na=False
#                 )

#                 if mask.any():
#                     bad_rows = df.loc[mask, "gene_name"]

#                     for idx, value in bad_rows.items():
#                         rows_missing_parentheses.append({
#                             "model": model,
#                             "question": question,
#                             "run_id": run_id,
#                             "row_index": idx,
#                             "value": value,
#                         })

In [8]:
rows_missing_parentheses = []

question_entities = {
    "gene_name": set(),
    "drug_name": set(),
    "disease_name": set(),
    "target_name": set(),
}

allowed_cols = {"gene_name", "drug_name", "disease_name", "target_name"}

for model, queries in dct_result.items():
    print(f"\nWorking for model: {model}")

    for question, runs in queries.items():
        for run_id, payload in runs.items():
            df = payload.get("dataframe")

            if not isinstance(df, pd.DataFrame) or df.empty:
                continue
            if "pathway_name" in df.columns:
                continue
            if not set(df.columns).issubset(allowed_cols):
                continue

            if "target_name" in df.columns:
                t = df["target_name"].astype("string")

                # validate only non-null rows
                bad_mask = t.notna() & ~t.str.contains(r"\([^()]+\)", regex=True)
                if bad_mask.any():
                    bad_rows = t[bad_mask].tolist()
                    raise ValueError(
                        f"target_name rows without parentheses: {bad_rows}"
                    )

                extracted_gene = t.str.extract(r"\(([^()]+)\)", expand=False).str.strip()

                # mutate original df in dict
                if "gene_name" in df.columns:
                    df["gene_name"] = df["gene_name"].astype("string").fillna(extracted_gene)
                else:
                    df["gene_name"] = extracted_gene

                if "gene_name" not in df.columns:
                    print("Issue found")

                # optional: if you want only normalized gene_name
                # df.drop(columns=["target_name"], inplace=True)

            for col in allowed_cols:
                if col in df.columns:
                    vals = df[col].dropna().astype(str).str.strip()
                    question_entities[col].update(v for v in vals if v)

# sets -> sorted lists
for k in question_entities:
    question_entities[k] = sorted(question_entities[k])



Working for model: ttd


##### Associate id with all gene entry

In [9]:
# Resolve unique gene symbols to IDs (MyGene -> OpenTargets fallback).
df_gene = resolve_ensembl_ids_with_fallback(
    list(question_entities["gene_name"]),
    use_opentargets_fallback=True,
)
df_gene = df_gene.drop_duplicates(subset=["gene_name", "gene_id"]).reset_index(drop=True)
df_gene


2026-02-27 18:50:55,679 [WARNING] Input sequence provided is already in string format. No operation performed
2026-02-27 18:50:55,680 [WARNING] Input sequence provided is already in string format. No operation performed
2026-02-27 18:50:55,681 [INFO] querying 1-757 ...
2026-02-27 18:50:57,820 [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-02-27 18:50:58,824 [INFO] Finished.
2026-02-27 18:50:58,924 [WARNING] 1 input query terms found dup hits:	[('GLRA4', 2)]
2026-02-27 18:50:58,924 [WARNING] 53 input query terms found no hit:	['ABL', 'APH1A-APH1B', 'BCR-ABL1', 'BACT FIMH', 'BACT FOLK', 'BACT FOLP', 'BACT GYRA', 'BACT GYRB', 
2026-02-27 18:50:58,930 [WARNING] Input sequence provided is already in string format. No operation performed
2026-02-27 18:50:58,931 [WARNING] Input sequence provided is already in string format. No operation performed
2026-02-27 18:50:58,931 [INFO] querying 1-32 ...
2026-02-27 18:50:59,228 [INFO] HTTP Request: POST https://mygene.in

[warn] Could not resolve 'APH1A-APH1B' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'BCR-ABL1' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'Bact FimH' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'Bact folK' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'Bact folP' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'Bact gyrA' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'Bact gyrB' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'Bact inhA' via MyGene or OpenTargets. Possible typo or ambiguous gene-family prefix.
[warn] Could not resolve 'CHRNA4-CHRNB2' via MyGene or OpenTargets. Possible typo or ambiguous gene-family pref

,gene_name,gene_id,source
0,AADAT,ENSG00000109576,mygene
1,ABAT,ENSG00000183044,mygene
2,ABCB1,ENSG00000085563,mygene
3,ABCB4,ENSG00000005471,mygene
4,ABCC9,ENSG00000069431,mygene
...,...,...,...
752,WEE1,ENSG00000166483,mygene
753,WT1,ENSG00000184937,mygene
754,XIAP,ENSG00000101966,mygene
755,XPO1,ENSG00000082898,mygene


In [10]:
df_gene[df_gene.isna().any(axis=1)]

,gene_name,gene_id,source
37,APH1A-APH1B,NaN,None
56,BCR-ABL1,NaN,None
67,Bact FimH,NaN,None
68,Bact folK,NaN,None
69,Bact folP,NaN,None
70,Bact gyrA,NaN,None
71,Bact gyrB,NaN,None
72,Bact inhA,NaN,None
146,CHRNA4-CHRNB2,NaN,None
185,Candi TMP1,NaN,None


##### Associate id with all drug entry

In [11]:
# Resolve unique drug surface forms to IDs.
# This keeps original raw names as rows and writes mapped ID + source.
df_drug = await get_chembl_ids_fast(list(question_entities["drug_name"]))
df_drug = df_drug.drop_duplicates(subset=["drug_name", "drug_id"]).reset_index(drop=True)
df_drug


2026-02-27 18:51:41,010 [INFO] [LLM] Total 845 names split into 11 batches (max_batch_chars=6000, max_batch_items=80, model=llama-3.3-70b-versatile).
2026-02-27 18:51:41,012 [INFO] [LLM] Batch size: 80
2026-02-27 18:51:41,053 [INFO] [LLM] Batch size: 80
2026-02-27 18:51:41,058 [INFO] [LLM] Batch size: 80
2026-02-27 18:51:41,061 [INFO] [LLM] Batch size: 80
2026-02-27 18:51:41,061 [INFO] [LLM] Batch size: 80
2026-02-27 18:51:41,062 [INFO] [LLM] Batch size: 80
2026-02-27 18:51:41,062 [INFO] [LLM] Batch size: 80
2026-02-27 18:51:41,063 [INFO] [LLM] Batch size: 80
2026-02-27 18:51:41,063 [INFO] [LLM] Batch size: 80
2026-02-27 18:51:41,063 [INFO] [LLM] Batch size: 80
2026-02-27 18:51:41,064 [INFO] [LLM] Batch size: 45


[map][drug][OpenTargets] raw='ABBV-154' -> id='CHEMBL5314652'
[map][drug][OpenTargets] raw='ABBV-257' -> id='CHEMBL4297682'
[map][drug][OpenTargets] raw='ABBV-323' -> id='CHEMBL4298212'
[map][drug][OpenTargets] raw='ABBV-951' -> id='CHEMBL4594379'
[map][drug][OpenTargets] raw='ABT-122' -> id='CHEMBL4297863'
[map][drug][OpenTargets] raw='ABT-874' -> id='CHEMBL1742995'
[map][drug][OpenTargets] raw='ADG116' -> id='CHEMBL5315007'
[map][drug][OpenTargets] raw='ADL-5859' -> id='CHEMBL494480'
[map][drug][OpenTargets] raw='ADX-48621' -> id='CHEMBL2346738'
[map][drug][OpenTargets] raw='AFM11' -> id='CHEMBL4297874'
[map][drug][OpenTargets] raw='AG-221' -> id='CHEMBL3989908'
[map][drug][OpenTargets] raw='AGEN1181' -> id='CHEMBL4650435'
[map][drug][OpenTargets] raw='AGEN1884' -> id='CHEMBL4297876'
[map][drug][OpenTargets] raw='AK104' -> id='CHEMBL4594456'
[map][drug][OpenTargets] raw='AKST4290' -> id='CHEMBL3670800'
[map][drug][OpenTargets] raw='ALD-403' -> id='CHEMBL3833320'
[map][drug][OpenTarge

2026-02-27 18:51:42,637 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:51:42,663 [INFO] [LLM] Batch done in 1.61s
2026-02-27 18:51:42,665 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:51:42,671 [INFO] [LLM] Batch done in 1.61s
2026-02-27 18:51:43,469 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:51:43,472 [INFO] [LLM] Batch done in 2.46s
2026-02-27 18:51:44,101 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:51:44,103 [INFO] [LLM] Batch done in 3.04s
2026-02-27 18:51:44,419 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:51:44,421 [INFO] [LLM] Batch done in 3.36s
2026-02-27 18:51:44,939 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:51:44

[map][drug][Llama] raw='Icotinib hydrochloride' -> canonical='ICOTINIB' -> id='CHEMBL2087361'


,drug_name,drug_id,source
0,(1-Benzyl-1H-indazol-5-yl)-quinazolin-4-yl-amine,None,None
1,(1-Benzyl-1H-indol-5-yl)-quinazolin-4-yl-amine,None,None
2,(3-Bromo-phenyl)-(5-nitro-quinazolin-4-yl)-amine,None,None
3,(3-Bromo-phenyl)-quinazolin-4-yl-amine,None,None
4,(E)-5-(4-Hydroxybenzylidene)-1-phenethylhydantoin,None,None
...,...,...,...
1344,Zavegepant,CHEMBL2397415,OpenTargets
1345,Zenocutuzomab,None,None
1346,Zolmitriptan,CHEMBL1185,OpenTargets
1347,Zunsemetinib,CHEMBL3704901,OpenTargets


In [12]:
df_drug["source"].value_counts()

source
OpenTargets    504
Llama            1
Name: count, dtype: int64

In [13]:
# df_drug[df_drug["source"]=="Llama"]

In [14]:
df_drug[df_drug.isna().any(axis=1)]

,drug_name,drug_id,source
0,(1-Benzyl-1H-indazol-5-yl)-quinazolin-4-yl-amine,None,None
1,(1-Benzyl-1H-indol-5-yl)-quinazolin-4-yl-amine,None,None
2,(3-Bromo-phenyl)-(5-nitro-quinazolin-4-yl)-amine,None,None
3,(3-Bromo-phenyl)-quinazolin-4-yl-amine,None,None
4,(E)-5-(4-Hydroxybenzylidene)-1-phenethylhydantoin,None,None
...,...,...,...
1338,Z-321,None,None
1339,ZN-e4,None,None
1341,ZT-1,None,None
1342,ZW49,None,None


##### Associate id with all disease entry

In [15]:
# Resolve unique disease surface forms to IDs.
# Canonical fallback is internal; returned rows remain keyed by raw input disease_name.
df_disease = await get_disease_ids_fast(list(question_entities["disease_name"]))
df_disease = df_disease.drop_duplicates(subset=["disease_name", "disease_id"]).reset_index(drop=True)
df_disease


2026-02-27 18:51:48,434 [INFO] [LLM] Total 34 names split into 1 batches (max_batch_chars=6000, max_batch_items=80, model=llama-3.3-70b-versatile).
2026-02-27 18:51:48,435 [INFO] [LLM] Batch size: 34


[map][disease][OpenTargets] raw='Acute myeloid leukaemia' -> id='MONDO_0015667'
[map][disease][OpenTargets] raw='Adrenal cancer' -> id='EFO_0003850'
[map][disease][OpenTargets] raw='Alzheimer disease' -> id='MONDO_0007088'
[map][disease][OpenTargets] raw='Angina pectoris' -> id='EFO_0003913'
[map][disease][OpenTargets] raw='Anxiety disorder' -> id='EFO_0006788'
[map][disease][OpenTargets] raw='Arterial occlusive disease' -> id='EFO_0003875'
[map][disease][OpenTargets] raw='Attention deficit hyperactivity disorder' -> id='EFO_0003888'
[map][disease][OpenTargets] raw='Autism spectrum disorder' -> id='EFO_0003756'
[map][disease][OpenTargets] raw='Binge eating disorder' -> id='EFO_0005924'
[map][disease][OpenTargets] raw='Bladder cancer' -> id='MONDO_0004986'
[map][disease][OpenTargets] raw='Brain cancer' -> id='MONDO_0001657'
[map][disease][OpenTargets] raw='Breast cancer' -> id='EFO_0000305'
[map][disease][OpenTargets] raw='Breast in situ carcinoma' -> id='MONDO_0004658'
[map][disease][O

2026-02-27 18:51:49,403 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:51:49,406 [INFO] [LLM] Batch done in 0.97s
2026-02-27 18:51:49,406 [INFO] [LLM] 'BCR-ABL1-negative chronic myeloid leukaemia' → 'chronic myeloid leukemia'
2026-02-27 18:51:49,407 [INFO] [LLM] 'Bone metastases' → None
2026-02-27 18:51:49,407 [INFO] [LLM] 'Cerebral ischaemia' → 'cerebral ischemia'
2026-02-27 18:51:49,408 [INFO] [LLM] 'Choreiform disorder' → None
2026-02-27 18:51:49,409 [INFO] [LLM] 'Dissociative neurological symptom disorder' → None
2026-02-27 18:51:49,409 [INFO] [LLM] 'Ejaculatory dysfunction' → None
2026-02-27 18:51:49,410 [INFO] [LLM] 'Epilepsy/seizure' → 'epilepsy'
2026-02-27 18:51:49,411 [INFO] [LLM] 'General pain disorder' → None
2026-02-27 18:51:49,411 [INFO] [LLM] 'HIV-infected patients with tuberculosis' → 'tuberculosis'
2026-02-27 18:51:49,411 [INFO] [LLM] 'Hepatitis virus infection' → 'hepatitis'
2026-02-27 18:51:49,412 [INFO] [LLM]

[map][disease][Llama] raw='BCR-ABL1-negative chronic myeloid leukaemia' -> canonical='chronic myelogenous leukemia' -> id='EFO_0000339'
[map][disease][Llama] raw='Cerebral ischaemia' -> canonical='cerebral infarction' -> id='EFO_0000712'
[map][disease][Llama] raw='Epilepsy/seizure' -> canonical='epilepsy' -> id='EFO_0000474'
[map][disease][Llama] raw='HIV-infected patients with tuberculosis' -> canonical='tuberculosis' -> id='MONDO_0018076'
[map][disease][Llama] raw='Hepatitis virus infection' -> canonical='Hepatitis' -> id='HP_0012115'
[map][disease][Llama] raw='Idiopathic parkinson disease' -> canonical='Parkinson disease' -> id='MONDO_0005180'
[map][disease][Llama] raw='Intracerebral haemorrhage' -> canonical='intracerebral hemorrhage' -> id='EFO_0005669'
[map][disease][Llama] raw='Multi-drug resistant tuberculosis' -> canonical='tuberculosis' -> id='MONDO_0018076'
[map][disease][Llama] raw='Recurrent glioblastoma' -> canonical='glioblastoma multiforme' -> id='EFO_0000519'
[warn] Un

,disease_name,disease_id,source
0,Acute myeloid leukaemia,MONDO_0015667,OpenTargets
1,Adrenal cancer,EFO_0003850,OpenTargets
2,Alzheimer disease,MONDO_0007088,OpenTargets
3,Angina pectoris,EFO_0003913,OpenTargets
4,Anxiety disorder,EFO_0006788,OpenTargets
...,...,...,...
133,Type 2 diabetes mellitus,MONDO_0005148,OpenTargets
134,Ulcerative colitis,EFO_0000729,OpenTargets
135,Unspecific body region injury,None,None
136,Urethral cancer,EFO_0003846,OpenTargets


In [16]:
df_disease["source"].value_counts()

source
OpenTargets    104
Llama            9
Name: count, dtype: int64

In [17]:
df_disease[df_disease.isna().any(axis=1)]

,disease_name,disease_id,source
11,Bone metastases,None,None
18,Choreiform disorder,None,None
36,Dissociative neurological symptom disorder,None,None
37,Ejaculatory dysfunction,None,None
45,General pain disorder,None,None
60,Lip/oral cavity/pharynx neoplasm,None,None
68,Malignant mesenchymal neoplasm,None,None
70,Mature B-cell leukaemia,None,None
71,Mature B-cell lymphoma,None,None
74,Menopausal disorder,None,None


In [18]:
# ------------------------------------------------------------------
# Disease canonical audit (OpenTargets-strict IDs only)
# ------------------------------------------------------------------
# IMPORTANT:
# - This cell runs an additional Llama canonicalization pass over unresolved rows.
# - IDs are still resolved ONLY via OpenTargets mapIds (score==1).
# - To avoid changing the baseline resolver output, writeback is OFF by default.
APPLY_AUDIT_MAPPINGS = False

# 1) Rows unresolved after primary resolver
unresolved_rows = df_disease[df_disease["disease_id"].isna()].reset_index(drop=True)
unresolved_raw_names = list(
    dict.fromkeys(
        str(name).strip()
        for name in unresolved_rows["disease_name"].dropna().astype(str).tolist()
        if str(name).strip()
    )
)

# 2) Llama canonicalization suggestions
disease_input_map = {name: None for name in unresolved_raw_names}
canonicalisation_map = await utility.canonicalise_disease_dict(disease_input_map)

# 3) OpenTargets strict verification (score==1 only)
canonical_terms_norm = list(
    dict.fromkeys(
        utility._normalize_disease_term(v)
        for v in canonicalisation_map.values()
        if isinstance(v, str) and v.strip()
    )
)
canonical_to_id = utility.resolve_diseases_opentargets_bulk(
    canonical_terms_norm,
    strict_score_one=True,
)

# 4) Build persistent audit table
audit_rows = []
applied_count = 0
candidate_count = 0

for raw_disease_name in unresolved_raw_names:
    canonical_name = canonicalisation_map.get(raw_disease_name)
    if isinstance(canonical_name, str) and canonical_name.strip():
        canonical_name = canonical_name.strip()
        canonical_norm = utility._normalize_disease_term(canonical_name)
        resolved_id = canonical_to_id.get(canonical_norm)
    else:
        canonical_name = None
        canonical_norm = None
        resolved_id = None

    can_apply = bool(resolved_id)
    if can_apply:
        candidate_count += 1
        print(f"[audit][disease][Llama->OT] raw='{raw_disease_name}' -> canonical='{canonical_name}' -> id='{resolved_id}'")

    if can_apply and APPLY_AUDIT_MAPPINGS:
        mask = df_disease["disease_name"].astype(str).str.strip() == raw_disease_name
        df_disease.loc[mask, "disease_id"] = resolved_id
        df_disease.loc[mask, "source"] = "Llama"
        applied_count += 1

    status = (
        "mappable_ot_exact" if can_apply else
        ("no_canonical" if canonical_name is None else "canonical_not_in_ot")
    )
    audit_rows.append({
        "raw_disease_name": raw_disease_name,
        "canonical_name": canonical_name,
        "canonical_norm": canonical_norm,
        "resolved_disease_id": resolved_id,
        "status": status,
        "can_apply": can_apply,
        "applied": bool(can_apply and APPLY_AUDIT_MAPPINGS),
    })

disease_canonical_audit_df = pd.DataFrame(audit_rows)
if not disease_canonical_audit_df.empty:
    disease_canonical_audit_df = disease_canonical_audit_df.sort_values(
        ["status", "raw_disease_name"],
        ascending=[True, True],
    ).reset_index(drop=True)

print(f"[summary][disease][audit] OT-mappable candidates: {candidate_count}")
print(f"[summary][disease][audit] applied to df_disease: {applied_count} (APPLY_AUDIT_MAPPINGS={APPLY_AUDIT_MAPPINGS})")
if not disease_canonical_audit_df.empty:
    print("[summary][disease][audit] status counts:")
    print(disease_canonical_audit_df["status"].value_counts(dropna=False))

disease_canonical_audit_df


2026-02-27 18:51:50,388 [INFO] [LLM] Total 25 names split into 1 batches (max_batch_chars=6000, max_batch_items=80, model=llama-3.3-70b-versatile).
2026-02-27 18:51:50,390 [INFO] [LLM] Batch size: 25
2026-02-27 18:51:51,335 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:51:51,338 [INFO] [LLM] Batch done in 0.95s
2026-02-27 18:51:51,339 [INFO] [LLM] 'Bone metastases' → None
2026-02-27 18:51:51,340 [INFO] [LLM] 'Choreiform disorder' → None
2026-02-27 18:51:51,340 [INFO] [LLM] 'Dissociative neurological symptom disorder' → None
2026-02-27 18:51:51,341 [INFO] [LLM] 'Ejaculatory dysfunction' → None
2026-02-27 18:51:51,341 [INFO] [LLM] 'General pain disorder' → None
2026-02-27 18:51:51,342 [INFO] [LLM] 'Lip/oral cavity/pharynx neoplasm' → None
2026-02-27 18:51:51,342 [INFO] [LLM] 'Malignant mesenchymal neoplasm' → None
2026-02-27 18:51:51,343 [INFO] [LLM] 'Mature B-cell leukaemia' → 'leukaemia'
2026-02-27 18:51:51,343 [INFO] [LLM] 'M

[audit][disease][Llama->OT] raw='Mature B-cell leukaemia' -> canonical='childhood leukemia' -> id='MONDO_0004355'
[audit][disease][Llama->OT] raw='Mature B-cell lymphoma' -> canonical='lymphoma' -> id='EFO_0000574'
[audit][disease][Llama->OT] raw='Non-small-cell lung cancer' -> canonical='non-small cell lung carcinoma' -> id='EFO_0003060'
[audit][disease][Llama->OT] raw='Subacute cutaneous lupus erythematosus' -> canonical='lupus erythematosus' -> id='MONDO_0004670'
[summary][disease][audit] OT-mappable candidates: 4
[summary][disease][audit] applied to df_disease: 0 (APPLY_AUDIT_MAPPINGS=False)
[summary][disease][audit] status counts:
status
no_canonical         21
mappable_ot_exact     4
Name: count, dtype: int64


,raw_disease_name,canonical_name,canonical_norm,resolved_disease_id,status,can_apply,applied
0,Mature B-cell leukaemia,childhood leukemia,childhood leukemia,MONDO_0004355,mappable_ot_exact,True,False
1,Mature B-cell lymphoma,lymphoma,lymphoma,EFO_0000574,mappable_ot_exact,True,False
2,Non-small-cell lung cancer,non-small cell lung carcinoma,non-small cell lung carcinoma,EFO_0003060,mappable_ot_exact,True,False
3,Subacute cutaneous lupus erythematosus,lupus erythematosus,lupus erythematosus,MONDO_0004670,mappable_ot_exact,True,False
4,Bone metastases,None,None,None,no_canonical,False,False
5,Choreiform disorder,None,None,None,no_canonical,False,False
6,Dissociative neurological symptom disorder,None,None,None,no_canonical,False,False
7,Ejaculatory dysfunction,None,None,None,no_canonical,False,False
8,General pain disorder,None,None,None,no_canonical,False,False
9,Lip/oral cavity/pharynx neoplasm,None,None,None,no_canonical,False,False


In [19]:
# disease_canonical_audit_df

In [20]:
df_disease[df_disease["source"]=='Llama']

,disease_name,disease_id,source
8,BCR-ABL1-negative chronic myeloid leukaemia,EFO_0000339,Llama
16,Cerebral ischaemia,EFO_0000712,Llama
39,Epilepsy/seizure,EFO_0000474,Llama
47,HIV-infected patients with tuberculosis,MONDO_0018076,Llama
50,Hepatitis virus infection,HP_0012115,Llama
54,Idiopathic parkinson disease,MONDO_0005180,Llama
57,Intracerebral haemorrhage,EFO_0005669,Llama
81,Multi-drug resistant tuberculosis,MONDO_0018076,Llama
113,Recurrent glioblastoma,EFO_0000519,Llama


In [21]:
df_disease["source"].value_counts()

source
OpenTargets    104
Llama            9
Name: count, dtype: int64

In [22]:
df_disease

,disease_name,disease_id,source
0,Acute myeloid leukaemia,MONDO_0015667,OpenTargets
1,Adrenal cancer,EFO_0003850,OpenTargets
2,Alzheimer disease,MONDO_0007088,OpenTargets
3,Angina pectoris,EFO_0003913,OpenTargets
4,Anxiety disorder,EFO_0006788,OpenTargets
...,...,...,...
133,Type 2 diabetes mellitus,MONDO_0005148,OpenTargets
134,Ulcerative colitis,EFO_0000729,OpenTargets
135,Unspecific body region injury,None,None
136,Urethral cancer,EFO_0003846,OpenTargets


In [23]:
# df_disease[df_disease["source"]=="Llama"]

In [24]:
def compute_jaccard_from_run_dataframes(dct_result, df_gene, df_disease, df_drug):
    """
    Compute per-question cross-run Jaccard consistency after mapping entities to IDs.

    For each (model, question):
    - intersection: entries present in ALL valid runs
    - union: entries present in ANY valid run
    - jaccard = |intersection| / |union|

    All comparisons are CASE-INDEPENDENT and ID-based.

    Side effects:
    - Stores per-run mapped dataframe in:
        dct_result[model][question][run]['dataframe_id']

    Returns:
    - dct_jaccard[model][question] = {
          'jaccard': float,
          'n_valid_runs': int,
          'intersection': set[tuple],
          'union': set[tuple],
      }
    """

    dct_jaccard = {}

    # ------------------------------------------------------------------
    # Build deterministic one-to-one mapping tables
    # ------------------------------------------------------------------
    gene_map = (
        df_gene.assign(_gene_norm=df_gene["gene_name"].astype(str).str.strip().str.upper())
        [["_gene_norm", "gene_id"]]
        .dropna(subset=["gene_id"])
        .drop_duplicates("_gene_norm", keep="first")
    )

    disease_map = (
        df_disease.assign(_disease_norm=df_disease["disease_name"].astype(str).str.strip().str.upper())
        [["_disease_norm", "disease_id"]]
        .dropna(subset=["disease_id"])
        .drop_duplicates("_disease_norm", keep="first")
    )

    drug_map = (
        df_drug.assign(_drug_norm=df_drug["drug_name"].astype(str).str.strip().str.upper())
        [["_drug_norm", "drug_id"]]
        .dropna(subset=["drug_id"])
        .drop_duplicates("_drug_norm", keep="first")
    )

    # ------------------------------------------------------------------
    # Main loop
    # ------------------------------------------------------------------
    for model, model_payload in dct_result.items():
        dct_jaccard[model] = {}
        print(f"\nWorking for model: {model}")

        for question_key, runs_dict in model_payload.items():
            run_sets = {}

            for run_number, run_payload in runs_dict.items():
                df = run_payload.get("dataframe")

                # ---- basic validity ----
                if not isinstance(df, pd.DataFrame) or df.empty:
                    continue

                if "pathway_name" in df.columns:
                    continue

                work_df = df.copy()

                # print(work_df.head())

                final_cols = []

                # ---- gene ----
                if "gene_name" in work_df.columns:
                    work_df["_gene_norm"] = (
                        work_df["gene_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(gene_map, on="_gene_norm", how="left")
                    final_cols.append("gene_id")

                # ---- disease ----
                if "disease_name" in work_df.columns:
                    work_df["_disease_norm"] = (
                        work_df["disease_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(disease_map, on="_disease_norm", how="left")
                    final_cols.append("disease_id")

                # ---- drug ----
                if "drug_name" in work_df.columns:
                    work_df["_drug_norm"] = (
                        work_df["drug_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(drug_map, on="_drug_norm", how="left")
                    final_cols.append("drug_id")

                if not final_cols:
                    continue

                # ---- canonical ID dataframe ----
                id_df = (
                    work_df[final_cols]
                    .dropna(how="any")
                    .drop_duplicates()
                    .reset_index(drop=True)
                )

                dct_result[model][question_key][run_number]["dataframe_id"] = id_df

                if id_df.empty:
                    continue

                # ---- CASE-INDEPENDENT tuple set ----
                run_set = {
                    tuple(str(v).strip().upper() for v in row)
                    for row in id_df.itertuples(index=False, name=None)
                }

                if run_set:
                    run_sets[run_number] = run_set

            # ------------------------------------------------------------------
            # Compute intersection / union
            # ------------------------------------------------------------------
            n_valid_runs = len(run_sets)

            if n_valid_runs < 2:
                print(
                    f"Not enough valid runs for '{question_key}' "
                    f"({n_valid_runs}/{len(runs_dict)}). Skipping."
                )
                continue

            sets = list(run_sets.values())
            intersection = set.intersection(*sets)
            union = set.union(*sets)

            if not union:
                print(f"Empty union for '{question_key}', skipping.")
                continue

            jaccard = len(intersection) / len(union)

            dct_jaccard[model][question_key] = {
                "jaccard": jaccard,
                "n_valid_runs": n_valid_runs,
                "intersection": intersection,
                "union": union,
            }

            print(
                f"'{question_key}': "
                f"J={jaccard:.4f}, "
                f"|∩|={len(intersection)}, "
                f"|∪|={len(union)}, "
                f"runs={n_valid_runs}"
            )

    return dct_jaccard


dct_jaccard = compute_jaccard_from_run_dataframes(
    dct_result=dct_result,
    df_gene=df_gene,
    df_disease=df_disease,
    df_drug=df_drug,
)


Working for model: ttd
'Which diseases are associated with BRAF?': J=1.0000, |∩|=4, |∪|=4, runs=5
Not enough valid runs for 'Which genes are associated with amyotrophic lateral sclerosis?' (0/0). Skipping.
'Which diseases are associated with CDK4?': J=1.0000, |∩|=9, |∪|=9, runs=5
Not enough valid runs for 'Which pathways are associated with the JAK2 gene?' (0/5). Skipping.
'Which drugs are used to treat rheumatoid arthritis?': J=1.0000, |∩|=140, |∪|=140, runs=5
'Which genes are associated with ovarian cancer?': J=1.0000, |∩|=83, |∪|=83, runs=5
'What are the known targets of regorafenib?': J=1.0000, |∩|=3, |∪|=3, runs=4
Not enough valid runs for 'Which pathways are associated with the STAT3 gene?' (0/5). Skipping.
'Which genes are associated with fever (pyrexia)?': J=0.2857, |∩|=2, |∪|=7, runs=5
'Which diseases are treated with bevacizumab?': J=1.0000, |∩|=3, |∪|=3, runs=5
'What are the known targets of cabozantinib?': J=1.0000, |∩|=2, |∪|=2, runs=5
Not enough valid runs for 'Which pat

In [25]:
# Run Jaccard consistency computation across runs after ID grounding.
dct_jaccard = compute_jaccard_from_run_dataframes(
    dct_result=dct_result,
    df_gene=df_gene,
    df_disease=df_disease,
    df_drug=df_drug,
)

# Flatten into a quick summary table for inspection.
rows = [
    {
        "model": model,
        "question": question,
        "jaccard": payload["jaccard"],
        "n_valid_runs": payload["n_valid_runs"],
        "intersection_size": len(payload["intersection"]),
        "union_size": len(payload["union"]),
    }
    for model, qmap in dct_jaccard.items()
    for question, payload in qmap.items()
]

jaccard_summary = pd.DataFrame(rows)
if jaccard_summary.empty:
    print("No valid Jaccard entries were produced. Check input runs and validation filters.")
else:
    display(jaccard_summary.sort_values(["model", "jaccard"], ascending=[True, False]).head(20))



Working for model: ttd
'Which diseases are associated with BRAF?': J=1.0000, |∩|=4, |∪|=4, runs=5
Not enough valid runs for 'Which genes are associated with amyotrophic lateral sclerosis?' (0/0). Skipping.
'Which diseases are associated with CDK4?': J=1.0000, |∩|=9, |∪|=9, runs=5
Not enough valid runs for 'Which pathways are associated with the JAK2 gene?' (0/5). Skipping.
'Which drugs are used to treat rheumatoid arthritis?': J=1.0000, |∩|=140, |∪|=140, runs=5
'Which genes are associated with ovarian cancer?': J=1.0000, |∩|=83, |∪|=83, runs=5
'What are the known targets of regorafenib?': J=1.0000, |∩|=3, |∪|=3, runs=4
Not enough valid runs for 'Which pathways are associated with the STAT3 gene?' (0/5). Skipping.
'Which genes are associated with fever (pyrexia)?': J=0.2857, |∩|=2, |∪|=7, runs=5
'Which diseases are treated with bevacizumab?': J=1.0000, |∩|=3, |∪|=3, runs=5
'What are the known targets of cabozantinib?': J=1.0000, |∩|=2, |∪|=2, runs=5
Not enough valid runs for 'Which pat

,model,question,jaccard,n_valid_runs,intersection_size,union_size
0,ttd,Which diseases are associated with BRAF?,1.0,5,4,4
1,ttd,Which diseases are associated with CDK4?,1.0,5,9,9
2,ttd,Which drugs are used to treat rheumatoid arthr...,1.0,5,140,140
3,ttd,Which genes are associated with ovarian cancer?,1.0,5,83,83
4,ttd,What are the known targets of regorafenib?,1.0,4,3,3
6,ttd,Which diseases are treated with bevacizumab?,1.0,5,3,3
7,ttd,What are the known targets of cabozantinib?,1.0,5,2,2
8,ttd,Which drugs are used to treat migraine?,1.0,5,35,35
9,ttd,Which genes are associated with epilepsy?,1.0,5,52,52
10,ttd,Which genes are associated with Alzheimer’s di...,1.0,5,144,144


In [26]:
# Persist enriched run payloads with dataframe_id attached for each run.
with open("TTD_same_question_response_with_id.pkl", "wb") as f:
    pickle.dump(dct_result, f)
